In [4]:
#COLLECTION_ROOT='/home/ec2-user/data/collections/'
COLLECTION_ROOT='/disk3/collections_cleaned/'
DATASET='LongEmbed'
DATASET_DIR=f'{COLLECTION_ROOT}/{DATASET}'
HF_DATASET='dwzhu/LongEmbed'
USE_CONFIGS=["passkey", "needle"]
MAX_QUERY_CONTEXT_LENGTH=32768

In [5]:
import os
DATASET_INPUT_DIR=f'{DATASET_DIR}/input_data/'
os.makedirs(DATASET_INPUT_DIR, exist_ok=True)

In [6]:
from datasets import load_dataset

In [7]:
from flexneuart.text_proc.parse import KrovetzStemParser

In [8]:
STOPWORDS="a about above according across after afterwards again against albeit all almost alone along already also although always am among amongst an and another any anybody anyhow anyone anything anyway anywhere apart are around as at av be became because become becomes becoming been before beforehand behind being below beside besides between beyond both but by can cannot canst certain cf choose contrariwise cos could cu day do does doesn't doing dost doth double down dual during each either else elsewhere enough et etc even ever every everybody everyone everything everywhere except excepted excepting exception exclude excluding exclusive far farther farthest few ff first for formerly forth forward from front further furthermore furthest get go had halves hardly has hast hath have he hence henceforth her here hereabouts hereafter hereby herein hereto hereupon hers herself him himself hindmost his hither hitherto how however howsoever i ie if in inasmuch inc include included including indeed indoors inside insomuch instead into inward inwards is it its itself just kind kg km last latter latterly less lest let like little ltd many may maybe me meantime meanwhile might moreover most mostly more mr mrs ms much must my myself namely need neither never nevertheless next no nobody none nonetheless noone nope nor not nothing notwithstanding now nowadays nowhere of off often ok on once one only onto or other others otherwise ought our ours ourselves out outside over own per perhaps plenty provide quite rather really round said sake same sang save saw see seeing seem seemed seeming seems seen seldom selves sent several shalt she should shown sideways since slept slew slung slunk smote so some somebody somehow someone something sometime sometimes somewhat somewhere spake spat spoke spoken sprang sprung stave staves still such supposing than that the thee their them themselves then thence thenceforth there thereabout thereabouts thereafter thereby therefore therein thereof thereon thereto thereupon these they this those thou though thrice through throughout thru thus thy thyself till to together too toward towards ugh unable under underneath unless unlike until up upon upward upwards us use used using very via vs want was we week well were what whatever whatsoever when whence whenever whensoever where whereabouts whereafter whereas whereat whereby wherefore wherefrom wherein whereinto whereof whereon wheresoever whereto whereunto whereupon wherever wherewith whether whew which whichever whichsoever while whilst whither who whoa whoever whole whom whomever whomsoever whose whosoever why will wilt with within without worse worst would wow ye yet year yippee you your yours yourself yourselves n't 'd 'll 'm 're 's 've".split()
#STOPWORDS

In [9]:
text_parser = KrovetzStemParser(STOPWORDS)

In [10]:
import json
from tqdm.auto import tqdm
from flexneuart.config import DOCID_FIELD, TEXT_RAW_FIELD_NAME, TEXT_FIELD_NAME, ANSWER_FILE_JSONL_GZ, QUESTION_FILE_JSON, QREL_FILE
from flexneuart.io import FileWrapper
from flexneuart.io.queries import write_queries_dict
from flexneuart.io.qrels import write_qrels_dict
from flexneuart.io.runs import write_run_dict
from collections import defaultdict
import numpy as np
np.random.seed(0)

doc_cl_dict = {}

for config in USE_CONFIGS:
    output_doc = []
    output_query_dict = {}
    config_pref = config + '_'
    output_subdir=f'{DATASET_INPUT_DIR}/{config}'

    all_doc_id_run = {}
    
    cl_arr = []

    for rec in load_dataset(HF_DATASET, config, split='corpus'):
        cl = rec['context_length']
        mod_doc_id = config_pref + rec['doc_id']
        output_doc.append({DOCID_FIELD: mod_doc_id,
                           TEXT_RAW_FIELD_NAME: rec['text'],
                           'context_length': rec['context_length'],
                           TEXT_FIELD_NAME: text_parser(rec['text'])})
        all_doc_id_run[mod_doc_id] = np.random.uniform()
        doc_cl_dict[mod_doc_id] = cl    

        cl_arr.append(cl)

    from collections import Counter, defaultdict
    print(Counter(cl_arr))

    cl_context_to_ids = defaultdict(list)    

    trec_runs = {}
    for rec in load_dataset(HF_DATASET, config, split='queries'):
        cl = rec['context_length']
        if cl <= MAX_QUERY_CONTEXT_LENGTH:
            mod_query_id = config_pref + rec['qid']
            trec_runs[mod_query_id] = all_doc_id_run
            output_query_dict[mod_query_id] = {DOCID_FIELD: mod_query_id,
                                               TEXT_RAW_FIELD_NAME: rec['text'],
                                               TEXT_FIELD_NAME: text_parser(rec['text'])}
            cl_context_to_ids[cl].append(mod_query_id)

    write_queries_dict(output_query_dict, f'{output_subdir}/{QUESTION_FILE_JSON}')
    write_run_dict(trec_runs, f'{DATASET_DIR}/derived_data/trec_runs_cached/{DATASET}/{config}/run.bz2')

    with FileWrapper(f'{output_subdir}/{ANSWER_FILE_JSONL_GZ}', 'w') as out_f:
        for jrec in output_doc:
            json.dump(jrec, out_f)
            out_f.write('\n')

    qrels_dict = defaultdict(dict)
    for rec in load_dataset(HF_DATASET, config, split='qrels'):
        if rec['context_length'] <= MAX_QUERY_CONTEXT_LENGTH:
            qrels_dict[config_pref + rec['qid']][config_pref + rec['doc_id']] = 1

    write_qrels_dict(qrels_dict, f'{output_subdir}/{QREL_FILE}')

    for cl, qids in cl_context_to_ids.items():
        query_dict_subset = {qid : output_query_dict[qid] for qid in qids}
        qrels_dict_subset = {qid : qrels_dict[qid] for qid in qids}
        tredc_runs_subset = {qid : trec_runs[qid] for qid in qids}

        write_queries_dict(query_dict_subset, f'{output_subdir}_{cl}/{QUESTION_FILE_JSON}')
        write_run_dict(tredc_runs_subset, f'{DATASET_DIR}/derived_data/trec_runs_cached/{DATASET}/{config}_{cl}/run.bz2')
        write_qrels_dict(qrels_dict_subset, f'{output_subdir}_{cl}/{QREL_FILE}')

Using the latest cached version of the dataset since dwzhu/LongEmbed couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'passkey' at /home/leo/.cache/huggingface/datasets/dwzhu___long_embed/passkey/0.0.0/10039a580487dacecf79db69166e17ace3ede392 (last modified on Thu Jan 30 07:03:28 2025).
Using the latest cached version of the dataset since dwzhu/LongEmbed couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'passkey' at /home/leo/.cache/huggingface/datasets/dwzhu___long_embed/passkey/0.0.0/10039a580487dacecf79db69166e17ace3ede392 (last modified on Thu Jan 30 07:03:28 2025).


Counter({256: 100, 512: 100, 1024: 100, 2048: 100, 4096: 100, 8192: 100, 16384: 100, 32768: 100})


Using the latest cached version of the dataset since dwzhu/LongEmbed couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'passkey' at /home/leo/.cache/huggingface/datasets/dwzhu___long_embed/passkey/0.0.0/10039a580487dacecf79db69166e17ace3ede392 (last modified on Thu Jan 30 07:03:28 2025).
Using the latest cached version of the dataset since dwzhu/LongEmbed couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'needle' at /home/leo/.cache/huggingface/datasets/dwzhu___long_embed/needle/0.0.0/10039a580487dacecf79db69166e17ace3ede392 (last modified on Thu Jan 30 06:48:35 2025).
Using the latest cached version of the dataset since dwzhu/LongEmbed couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'needle' at /home/leo/.cache/huggingface/datasets/dwzhu___long_embed/needle/0.0.0/10039a580487dacecf79db69166e17ace3ede392 (last modified on Thu Jan 30 06:48:35 2025).


Counter({256: 100, 512: 100, 1024: 100, 2048: 100, 4096: 100, 8192: 100, 16384: 100, 32768: 100})


Using the latest cached version of the dataset since dwzhu/LongEmbed couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'needle' at /home/leo/.cache/huggingface/datasets/dwzhu___long_embed/needle/0.0.0/10039a580487dacecf79db69166e17ace3ede392 (last modified on Thu Jan 30 06:48:35 2025).


In [11]:
cl_context_to_ids.keys()

dict_keys([256, 512, 1024, 2048, 4096, 8192, 16384, 32768])